# Fake/Real News Classifier — Full-Dataset, Fixed-Qubit Variational Quantum Classifier

**How this differs from the earlier DisCoCat/lambeq notebook, and why:**
- Trains on the **full training split (~27k documents)**, not a 300-doc subset.
- That's only possible because qubit count is **fixed** here (12 qubits, always), instead of
  scaling with word/sentence count as it did with the lambeq/DisCoCat ansatz. Text is reduced
  to a small fixed-length embedding *before* it reaches the quantum circuit. This is the same
  trade-off flagged earlier: this is a **hybrid classical-embedding + quantum-classifier**
  model, not "pure QNLP" — that's the price of full-dataset scale.
- **All classification weights are quantum.** The trainable parameters live in a PennyLane
  `StronglyEntanglingLayers` variational circuit acting on angle-encoded features. The only
  classical learned parameters are a scalar scale and bias used to calibrate the circuit's
  output into a probability — everything that "decides" fake vs. real happens inside the
  quantum circuit.
- **Leakage mitigation.** This dataset's real-news half is almost entirely Reuters wire copy
  that opens with a dateline like `WASHINGTON (Reuters) -`; the fake-news half essentially
  never has this. A model can hit very high accuracy just by detecting that formatting, not by
  detecting misinformation. This notebook strips datelines and explicit source mentions before
  feature extraction, so results reflect content-based classification rather than a
  wire-service detector.

**Read before you run this — honest expectations:**
- **I have not executed this notebook.** The raw `Fake.csv` / `True.csv` files aren't available
  in the sandbox this was built in, and a full ~27k-document training run takes real wall-clock
  time even with a small fixed circuit. You need to run this yourself (Kaggle/Colab/local) to
  get real numbers.
- **The >90% accuracy target may be in tension with genuine fake-news detection on this
  dataset.** Once datelines/source mentions are stripped, you're left with a real but *smaller
  and less certain* signal. It may land above or below 90% depending on how much "fakeness" is
  actually recoverable from remaining content (word choice, structure, sourcing language,
  hedging, etc.) — I've built this to maximize genuine signal (full dataset, TF-IDF with
  bigrams, SVD-reduced embedding, a reasonably deep circuit), but I can't guarantee a number
  without running it. Treat >90% as a stretch goal, not a promise.
- **If you still see >95%+ accuracy after stripping**, that's a signal there's a *different*
  leftover leakage source (site-specific boilerplate, formatting quirks, etc.) worth
  investigating manually — not proof the model has learned to detect misinformation.
- Section 9 includes a quick classical sanity check comparing stripped vs. unstripped text, so
  you can see empirically how much the dateline was contributing on your own run.


## 1. Setup, logging, reproducibility

In [1]:
!pip install -q -U pennylane scikit-learn scipy torch


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

In [2]:
import torch
import random
import numpy as np
import logging

SEED = 12
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    force=True,
)
logger = logging.getLogger("vqc_fake_news")


## 2. Config

In [3]:
# --------------------------------------------------------------
# CONFIG
# --------------------------------------------------------------

# Quantum circuit -- FIXED size, independent of document length
N_QUBITS = 12                    # feature/embedding dimension == number of qubits
N_LAYERS = 6                     # StronglyEntanglingLayers depth
DIFF_METHOD = "best"             # PennyLane picks the fastest differentiable method available

# Feature extraction (classical, fixed-length -- this is what makes full-dataset training possible)
TFIDF_MAX_FEATURES = 30000
TFIDF_NGRAM_RANGE = (1, 2)

# Dataset size -- USE THE FULL TRAIN SPLIT (no 300-doc cap like the DisCoCat notebook)
USE_FULL_TRAIN = True            # set False + N_TRAIN_SUBSET to debug on a smaller slice first
N_TRAIN_SUBSET = 2000            # only used if USE_FULL_TRAIN is False

# Training
BATCH_SIZE = 128
EPOCHS = 15
LEARNING_RATE = 0.01
MODEL_CHECKPOINT_PATH = "vqc_fake_news_best.pt"
MODEL_FINAL_PATH = "vqc_fake_news_final.pt"

# --- fail fast on bad config ---
assert N_QUBITS > 0 and N_LAYERS > 0
assert TFIDF_MAX_FEATURES > N_QUBITS, "need more TF-IDF features than qubits for SVD to reduce meaningfully"
assert BATCH_SIZE > 0 and EPOCHS > 0 and LEARNING_RATE > 0

logger.info(
    "Config OK: N_QUBITS=%d N_LAYERS=%d TFIDF_MAX_FEATURES=%d USE_FULL_TRAIN=%s "
    "BATCH_SIZE=%d EPOCHS=%d LR=%.4f",
    N_QUBITS, N_LAYERS, TFIDF_MAX_FEATURES, USE_FULL_TRAIN, BATCH_SIZE, EPOCHS, LEARNING_RATE,
)


2026-08-26 21:18:45,365 [INFO] Config OK: N_QUBITS=12 N_LAYERS=6 TFIDF_MAX_FEATURES=30000 USE_FULL_TRAIN=True BATCH_SIZE=128 EPOCHS=15 LR=0.0100


## 3. Load and clean the dataset (full splits, no subset caps)

In [4]:
import glob
import pandas as pd

def _find_csv(name):
    matches = glob.glob(f"**/{name}", recursive=True)
    return matches[0] if matches else f"/kaggle/input/datasets/clmentbisaillon/fake-and-real-news-dataset/{name}"


def load_fake_real_news():
    fake = pd.read_csv(_find_csv("Fake.csv"))
    real = pd.read_csv(_find_csv("True.csv"))
    fake["label"] = 0   # Fake news
    real["label"] = 1   # Real news
    df = pd.concat([fake, real], ignore_index=True)
    df["content"] = df["title"].fillna("") + " " + df["text"].fillna("")
    return df[["content", "label"]]


df_final = load_fake_real_news()
logger.info("Raw dataset shape: %s", df_final.shape)
logger.info("Label distribution:\n%s", df_final["label"].value_counts().to_string())


2026-08-26 21:18:45,848 [INFO] NumExpr defaulting to 4 threads.
2026-08-26 21:18:48,949 [INFO] Raw dataset shape: (44898, 2)
2026-08-26 21:18:48,959 [INFO] Label distribution:
label
0    23481
1    21417


In [5]:
df = df_final.copy()
df["content"] = df["content"].fillna("").astype(str).str.strip()
df = df[df["content"] != ""]
df = df.drop_duplicates(subset=["content"])
df["label"] = df["label"].astype(int)
logger.info("Cleaned dataset shape: %s", df.shape)


2026-08-26 21:18:49,208 [INFO] Cleaned dataset shape: (39103, 2)


In [6]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["label"])

if not USE_FULL_TRAIN:
    train_df = train_df.sample(n=min(N_TRAIN_SUBSET, len(train_df)), random_state=42)

logger.info("Train: %s | Validation: %s | Test: %s", train_df.shape, val_df.shape, test_df.shape)

train_texts, train_labels = train_df["content"].tolist(), train_df["label"].tolist()
val_texts, val_labels = val_df["content"].tolist(), val_df["label"].tolist()
test_texts, test_labels = test_df["content"].tolist(), test_df["label"].tolist()


2026-08-26 21:18:50,830 [INFO] Train: (27372, 2) | Validation: (5865, 2) | Test: (5866, 2)


## 4. Leakage mitigation — strip datelines & explicit source mentions

This is the step the earlier DisCoCat notebook didn't have. Without it, the model can hit very
high accuracy by detecting `CITY (Reuters) -` formatting rather than anything about the article's
truthfulness. This is a **partial** mitigation (dateline pattern + explicit "Reuters" mentions) —
it will not catch every stylistic leak (e.g. other wire-service conventions, site-specific
boilerplate). Treat this as reducing the leak, not proving it's gone; section 9 gives you a way
to check empirically on your own run.

In [7]:
import re

_DATELINE_RE = re.compile(
    r'^[A-Z][A-Za-z.,\'\-\s]{0,40}\(Reuters\)\s*-\s*', re.MULTILINE
)
_REUTERS_MENTION_RE = re.compile(r'\breuters\b', re.IGNORECASE)

def strip_leakage(text):
    text = _DATELINE_RE.sub('', text)
    text = _REUTERS_MENTION_RE.sub('[source]', text)
    return text.strip()

n_dateline_hits = sum(bool(_DATELINE_RE.search(t)) for t in train_texts)
logger.info("Datelines found & stripped in %d / %d train docs (%.1f%%)",
            n_dateline_hits, len(train_texts), 100 * n_dateline_hits / max(len(train_texts), 1))

example_before = next(t for t in train_texts if _DATELINE_RE.search(t))
example_after = strip_leakage(example_before)
logger.info("Example before: %s", example_before[:160])
logger.info("Example after:  %s", example_after[:160])

train_texts_clean = [strip_leakage(t) for t in train_texts]
val_texts_clean = [strip_leakage(t) for t in val_texts]
test_texts_clean = [strip_leakage(t) for t in test_texts]


2026-08-26 21:18:51,608 [INFO] Datelines found & stripped in 11 / 27372 train docs (0.0%)
2026-08-26 21:18:51,685 [INFO] Example before: Barron's endorses Kasich for president (Reuters) - Calling him “the best hope for investors who want a Republican in the White House,” Barron’s announced its su
2026-08-26 21:18:51,686 [INFO] Example after:  Calling him “the best hope for investors who want a Republican in the White House,” Barron’s announced its support for Ohio Governor John Kasich for president w


## 5. Fixed-length feature extraction (TF-IDF → SVD → qubit-sized vector)

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler
import numpy as np

vectorizer = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=TFIDF_NGRAM_RANGE,
    stop_words="english",
)
train_tfidf = vectorizer.fit_transform(train_texts_clean)
val_tfidf = vectorizer.transform(val_texts_clean)
test_tfidf = vectorizer.transform(test_texts_clean)
logger.info("TF-IDF vocab size: %d", len(vectorizer.vocabulary_))

svd = TruncatedSVD(n_components=N_QUBITS, random_state=SEED)
train_embed = svd.fit_transform(train_tfidf)
val_embed = svd.transform(val_tfidf)
test_embed = svd.transform(test_tfidf)
logger.info("SVD explained variance ratio (sum): %.4f", svd.explained_variance_ratio_.sum())

# Scale to [0, pi] for angle encoding
scaler = MinMaxScaler(feature_range=(0, np.pi))
train_embed = scaler.fit_transform(train_embed)
val_embed = scaler.transform(val_embed)
test_embed = scaler.transform(test_embed)

logger.info("Feature matrix shapes -- train: %s val: %s test: %s",
            train_embed.shape, val_embed.shape, test_embed.shape)


2026-08-26 21:19:29,514 [INFO] TF-IDF vocab size: 30000
2026-08-26 21:19:31,484 [INFO] SVD explained variance ratio (sum): 0.0514
2026-08-26 21:19:31,492 [INFO] Feature matrix shapes -- train: (27372, 12) val: (5865, 12) test: (5866, 12)


## 6. Quantum variational classifier (PennyLane, fixed 12-qubit circuit)

In [9]:
import pennylane as qml

try:
    dev = qml.device("lightning.qubit", wires=N_QUBITS)
    logger.info("Using lightning.qubit simulator (fast C++ backend)")
except Exception as e:
    dev = qml.device("default.qubit", wires=N_QUBITS)
    logger.warning("lightning.qubit unavailable (%s), falling back to default.qubit", e)

@qml.qnode(dev, interface="torch", diff_method=DIFF_METHOD)
def circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return qml.expval(qml.PauliZ(0))

weight_shapes = {"weights": (N_LAYERS, N_QUBITS, 3)}
qlayer = qml.qnn.TorchLayer(circuit, weight_shapes)

logger.info("Quantum circuit: %d qubits, %d layers, %d trainable circuit parameters",
            N_QUBITS, N_LAYERS, N_LAYERS * N_QUBITS * 3)


2026-08-26 21:19:36,290 [INFO] Using lightning.qubit simulator (fast C++ backend)
2026-08-26 21:19:36,294 [INFO] Quantum circuit: 12 qubits, 6 layers, 216 trainable circuit parameters


In [10]:
import torch.nn as nn

class VQCClassifier(nn.Module):
    """All classification capacity lives in `qlayer` (the quantum circuit).
    `scale`/`bias` are a single calibration pair, not a hidden classical model."""
    def __init__(self, qlayer):
        super().__init__()
        self.qlayer = qlayer
        self.scale = nn.Parameter(torch.tensor(1.0, dtype=torch.float64))
        self.bias = nn.Parameter(torch.tensor(0.0, dtype=torch.float64))

    def forward(self, x):
        z = self.qlayer(x)                      # expval in [-1, 1]
        return torch.sigmoid(self.scale * z + self.bias)   # -> probability of "real" (label 1)

model = VQCClassifier(qlayer).double()
n_quantum_params = sum(p.numel() for n, p in model.named_parameters() if "qlayer" in n)
n_classical_params = sum(p.numel() for n, p in model.named_parameters() if "qlayer" not in n)
logger.info("Trainable params -- quantum: %d | classical (scale+bias only): %d",
            n_quantum_params, n_classical_params)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


2026-08-26 21:19:36,390 [INFO] Trainable params -- quantum: 216 | classical (scale+bias only): 2


## 7. Training loop over the full training set

Minibatched (not per-word circuits like the DisCoCat notebook), so document count no longer
drives circuit size -- only qubit count and layer depth do, and those are fixed.

In [11]:
import time

train_X = torch.tensor(train_embed, dtype=torch.float64)
train_y = torch.tensor(train_labels, dtype=torch.float64)
val_X = torch.tensor(val_embed, dtype=torch.float64)
val_y = torch.tensor(val_labels, dtype=torch.float64)
test_X = torch.tensor(test_embed, dtype=torch.float64)
test_y = torch.tensor(test_labels, dtype=torch.float64)

def evaluate(X, y):
    model.eval()
    with torch.no_grad():
        preds = model(X)
        acc = (torch.round(preds) == y).float().mean().item()
    model.train()
    return acc

best = {"acc": 0.0, "epoch": 0}
n_train = train_X.shape[0]
training_start = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()
    perm = torch.randperm(n_train)
    epoch_loss, n_batches = 0.0, 0

    for start in range(0, n_train, BATCH_SIZE):
        idx = perm[start:start + BATCH_SIZE]
        xb, yb = train_X[idx], train_y[idx]

        optimizer.zero_grad()
        preds = model(xb)
        loss = torch.nn.functional.binary_cross_entropy(preds, yb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    epoch_time = time.time() - epoch_start
    avg_loss = epoch_loss / max(n_batches, 1)
    if epoch == 0:
        logger.info("First epoch took %.1fs over %d batches -- extrapolate: %d epochs ~= %.1f min total",
                     epoch_time, n_batches, EPOCHS, epoch_time * EPOCHS / 60)

    val_acc = evaluate(val_X, val_y)
    logger.info("Epoch %d | train_loss=%.4f | val_acc=%.4f | epoch_time=%.1fs",
                 epoch, avg_loss, val_acc, epoch_time)

    if val_acc > best["acc"]:
        best["acc"] = val_acc
        best["epoch"] = epoch
        torch.save(model.state_dict(), MODEL_CHECKPOINT_PATH)
        logger.info("New best val_acc=%.4f -- checkpoint saved", val_acc)

logger.info("Training complete in %.1f min. Best val_acc=%.4f (epoch %d)",
            (time.time() - training_start) / 60, best["acc"], best["epoch"])

if best["acc"] > 0:
    model.load_state_dict(torch.load(MODEL_CHECKPOINT_PATH))
    logger.info("Reloaded best checkpoint for evaluation")


2026-08-26 21:41:37,347 [INFO] First epoch took 1316.8s over 214 batches -- extrapolate: 15 epochs ~= 329.2 min total
2026-08-26 21:44:59,104 [INFO] Epoch 0 | train_loss=0.5832 | val_acc=0.8749 | epoch_time=1316.8s
2026-08-26 21:44:59,110 [INFO] New best val_acc=0.8749 -- checkpoint saved
2026-08-26 22:08:47,473 [INFO] Epoch 1 | train_loss=0.3637 | val_acc=0.9055 | epoch_time=1246.3s
2026-08-26 22:08:47,476 [INFO] New best val_acc=0.9055 -- checkpoint saved
2026-08-26 22:31:22,657 [INFO] Epoch 2 | train_loss=0.2977 | val_acc=0.8996 | epoch_time=1174.9s
2026-08-26 22:54:07,717 [INFO] Epoch 3 | train_loss=0.2669 | val_acc=0.9134 | epoch_time=1186.0s
2026-08-26 22:54:07,719 [INFO] New best val_acc=0.9134 -- checkpoint saved
2026-08-26 23:16:55,761 [INFO] Epoch 4 | train_loss=0.2472 | val_acc=0.9183 | epoch_time=1187.3s
2026-08-26 23:16:55,763 [INFO] New best val_acc=0.9183 -- checkpoint saved
2026-08-26 23:39:14,278 [INFO] Epoch 5 | train_loss=0.2338 | val_acc=0.9207 | epoch_time=1160.6s


## 8. Test evaluation

In [12]:
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, accuracy_score

model.eval()
with torch.no_grad():
    test_preds = model(test_X)
test_pred_labels = torch.round(test_preds).long().tolist()

acc = accuracy_score(test_labels, test_pred_labels)
precision, recall, f1, _ = precision_recall_fscore_support(
    test_labels, test_pred_labels, average="binary", zero_division=0
)
cm = confusion_matrix(test_labels, test_pred_labels)

logger.info("Test accuracy:  %.4f", acc)
logger.info("Test precision: %.4f", precision)
logger.info("Test recall:    %.4f", recall)
logger.info("Test F1:        %.4f", f1)
logger.info("Confusion matrix [[TN, FP], [FN, TP]]:\n%s", cm)

if acc < 0.90:
    logger.warning(
        "Test accuracy is below the 90%% target. Given the leakage stripping in section 4, "
        "this is a plausible honest outcome, not necessarily a bug -- see the notebook header."
    )


2026-08-27 03:05:03,057 [INFO] Test accuracy:  0.9223
2026-08-27 03:05:03,058 [INFO] Test precision: 0.9313
2026-08-27 03:05:03,059 [INFO] Test recall:    0.9248
2026-08-27 03:05:03,060 [INFO] Test F1:        0.9281
2026-08-27 03:05:03,060 [INFO] Confusion matrix [[TN, FP], [FN, TP]]:
[[2469  217]
 [ 239 2941]]


## 9. Leakage sanity check (diagnostic, not the main model)

A fast classical baseline (logistic regression on raw TF-IDF) run **with** and **without**
dateline stripping. If accuracy drops sharply once stripped, that's direct evidence of how much
the earlier (and this) model could have been leaning on formatting rather than content. This is
a diagnostic tool for you to sanity-check the run above, not a replacement for it.

In [13]:
from sklearn.linear_model import LogisticRegression

def quick_baseline_acc(train_texts_variant, test_texts_variant, label):
    vec = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES, ngram_range=(1, 2), stop_words="english")
    Xtr = vec.fit_transform(train_texts_variant)
    Xte = vec.transform(test_texts_variant)
    clf = LogisticRegression(max_iter=200)
    clf.fit(Xtr, train_labels)
    acc = clf.score(Xte, test_labels)
    logger.info("Classical baseline (%s) test accuracy: %.4f", label, acc)
    return acc

acc_unstripped = quick_baseline_acc(train_texts, test_texts, "unstripped, with datelines")
acc_stripped = quick_baseline_acc(train_texts_clean, test_texts_clean, "stripped, leakage-mitigated")

logger.info("Leakage gap (unstripped - stripped): %.4f", acc_unstripped - acc_stripped)


2026-08-27 03:05:32,176 [INFO] Classical baseline (unstripped, with datelines) test accuracy: 0.9857
2026-08-27 03:06:01,236 [INFO] Classical baseline (stripped, leakage-mitigated) test accuracy: 0.9841
2026-08-27 03:06:01,271 [INFO] Leakage gap (unstripped - stripped): 0.0015


## 10. Save final model & inference function

In [14]:
import joblib

torch.save(model.state_dict(), MODEL_FINAL_PATH)
joblib.dump({"vectorizer": vectorizer, "svd": svd, "scaler": scaler}, "vqc_fake_news_pipeline.joblib")
logger.info("Model saved to %s, feature pipeline saved to vqc_fake_news_pipeline.joblib", MODEL_FINAL_PATH)


2026-08-27 03:06:01,902 [INFO] Model saved to vqc_fake_news_final.pt, feature pipeline saved to vqc_fake_news_pipeline.joblib


In [15]:
def predict_fake_or_real(text: str) -> dict:
    """Run a single new article through the trained pipeline."""
    cleaned = strip_leakage(str(text))
    tfidf_vec = vectorizer.transform([cleaned])
    embed = svd.transform(tfidf_vec)
    embed = scaler.transform(embed)
    x = torch.tensor(embed, dtype=torch.float64)

    model.eval()
    with torch.no_grad():
        p_real = model(x).item()
    model.train()

    label = "real" if p_real >= 0.5 else "fake"
    confidence = p_real if p_real >= 0.5 else 1 - p_real
    return {"label": label, "confidence": confidence, "p_real": p_real}


example = test_texts[0] if test_texts else "Officials confirmed the policy today."
result = predict_fake_or_real(example)
logger.info("Sample inference: %s", result)
print(result)


2026-08-27 03:06:02,034 [INFO] Sample inference: {'label': 'real', 'confidence': 0.5381692594875275, 'p_real': 0.5381692594875275}


{'label': 'real', 'confidence': 0.5381692594875275, 'p_real': 0.5381692594875275}
